In [1]:
import os
from pathlib import Path
from dotenv import dotenv_values, load_dotenv

env_path = Path("/Users/daniel/code/lca-lc-foundations/.env")
file_values = dotenv_values(env_path)

load_dotenv(env_path, override=True)

loaded_key = os.environ.get("LANGSMITH_API_KEY", "")
file_key = file_values.get("LANGSMITH_API_KEY", "")

print("Using endpoint:", os.environ.get("LANGSMITH_ENDPOINT"))
print("Key matches .env:", loaded_key == file_key)

Using endpoint: https://apac.api.smith.langchain.com
Key matches .env: True


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [4]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is seeking information about Lunapolis, the capital of the moon, including its weather, population, and current social issues affecting its residents.\n\n## SUMMARY\nThe capital of the moon is Lunapolis. The current weather in Lunapolis features clear skies, with temperatures ranging from a high of 120°C to a low of -100°C. Lunapolis has a population of 100,000 cheese miners. There is a possibility of a strike by the cheese miners' union due to discontent with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='f067b0af-a764-4728-8b11-0b80f5ee3843'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='9130a887-6c04-4e64-9036-14988d7eb518'),
              AIMessage(c

In [7]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is seeking information about Lunapolis, the capital of the moon, including its weather, population, and current social issues affecting its residents.

## SUMMARY
The capital of the moon is Lunapolis. The current weather in Lunapolis features clear skies, with temperatures ranging from a high of 120°C to a low of -100°C. Lunapolis has a population of 100,000 cheese miners. There is a possibility of a strike by the cheese miners' union due to discontent with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [8]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [9]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [10]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='460d33cc-58b4-409c-a81b-1e992e81e188'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='f0dbb2c8-392f-4512-b77d-43b0da679fdb', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='3fb95be4-4555-476c-98fa-a321715836b5'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='beb7091b-4fdf-4f56-b629-822e42b54624', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='de43d381-abb3-48fe-aa90-8c27ea90d5fb'),
              AIMessage(content='I can’t read your device’s temperature from here. If you want 

In [11]:
print(response["messages"][-1].content)

I can’t read your device’s temperature from here. If you want to know the exact temperature, tell me what type of device you’re using (PC/laptop, smartphone, tablet, etc.) and the OS/model, and I’ll give you precise steps to check it.

In the meantime, some quick safety and troubleshooting tips:
- If the device feels very hot to the touch, smells burnt, or the fan is loud, power it off, unplug it, and let it cool on a nonflammable surface with good airflow.
- Check for blocked vents and clear dust from the intake/exhaust.
- If you have a removable battery, power down and remove the battery (only if safe to do so) before inspecting.
- After cooling, try turning it on again. If it still won’t start, note any indicator lights or beeps as they can help identify the problem.

If you want, share your device type and OS and I’ll guide you through checking the temperature step by step.
